In [ ]:
# A100 GPU 환경에 맞춘 필수 라이브러리 설치
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate peft bitsandbytes qwen-vl-utils
!pip install -q flash-attn --no-build-isolation

import os
import re
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from qwen_vl_utils import process_vision_info

In [ ]:
# 1. 프롬프트 정의
SYSTEM_INSTRUCT = (
    "당신은 재활용품 이미지 기반 객관식 VQA 어시스턴트입니다. "
    "질문과 보기를 먼저 읽고, 이미지에서 확인해야 할 핵심 객체와 속성을 찾으세요. "
    "반드시 이미지와 보기의 일치 여부를 비교하여 가장 적절한 선택지를 고르세요. "
    "질문에 이미 답이 직접 포함된 경우에는 이미지보다 질문의 의미를 우선 해석하세요. "
    "출력은 반드시 a, b, c, d 중 하나의 소문자 한 글자만 하세요. "
    "설명, 공백, 문장, 부호는 절대 출력하지 마세요."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        "다음은 재활용품 이미지에 대한 객관식 문제입니다.\n"
        "먼저 질문과 보기를 읽고, 사진에서 찾아야 할 대상 객체와 판단 기준을 정하세요.\n\n"
        f"질문: {question}\n\n"
        "보기:\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "판단 규칙:\n"
        "1. 먼저 문제와 보기를 보고 사진에서 찾아야 할 핵심 객체를 정하세요.\n"
        "2. 객체 개수를 묻는 문제라면, 동일한 객체가 2개 이상 있는지 세고, 정확하지 않으면 보기 중 가장 가까운 개수를 고르세요.\n"
        "3. 객체 종류를 묻는 문제라면, 보기 후보들을 이미지와 대조하여 사진 묘사와 다른 보기는 제외하세요.\n"
        "4. 정답 후보가 2개 이상으로 보이면, 해당 객체들의 개수를 비교하여 더 많이 보이는 객체 쪽을 우선 선택하세요.\n"
        "5. 사진을 찍은 사람의 시점에서, 가리키는 대상이나 중심 대상이 무엇인지 추정하세요.\n"
        "6. 컵과 뚜껑은 반드시 구분하세요. 컵인지, 뚜껑인지, 컵의 일부인지 주의해서 판단하세요.\n"
        "7. 플라스틱 컵이 여러 개 겹쳐 있으면, 적층된 정도를 보고 컵 개수를 추정하세요.\n"
        "8. 사진 속 물건에 글자가 보이면, 그 글자를 단서로 물건의 종류와 재질을 판단하세요.\n"
        "9. 물건의 재질은 금속(캔), 유리, 플라스틱, 종이(골판지 포함) 중 무엇에 가까운지 판단하세요.\n"
        "10. 컵라면 용기처럼 질문 자체에 답의 단서가 있는 경우에는 이미지보다 질문의 의미를 우선 반영하세요.\n"
        "11. 사진을 보고 대상의 색깔, 형태, 로고, 뚜껑 여부, 라벨 여부 등을 내부적으로 묘사한 뒤 보기와 비교하세요.\n"
        "12. 사진 묘사와 명확히 다른 보기는 정답 후보에서 제외하세요.\n"
        "13. 재질이 불확실하면 플라스틱을 우선 고려하세요. 스티로폼도 플라스틱 범주로 간주하세요.\n"
        "14. 최종적으로 가장 적절한 보기 하나를 선택하세요.\n\n"
        "출력 규칙:\n"
        "- 반드시 a, b, c, d 중 하나의 소문자 한 글자만 출력하세요.\n"
        "- 설명하지 마세요.\n"
        "- 다른 글자나 문장을 절대 출력하지 마세요.\n\n"
        "정답:"
    )

# 2. 커스텀 Dataset 클래스
class QwenVQADataset(Dataset):
    def __init__(self, df, img_dir, processor, is_train=True):
        self.df = df
        self.img_dir = img_dir
        self.processor = processor
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['path'])
        image = Image.open(img_path).convert("RGB")
        
        user_text = build_mc_prompt(row['question'], row['a'], row['b'], row['c'], row['d'])
        
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": user_text}]}
        ]
        
        if self.is_train and 'answer' in row:
            messages.append({"role": "assistant", "content": [{"type": "text", "text": row['answer'].strip().lower()}]})
            
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=not self.is_train)
        
        # Qwen-VL 프로세서를 통한 텐서 변환
        inputs = self.processor(
            text=[text],
            images=[image],
            padding=True,
            return_tensors="pt"
        )
        
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        
        # 라벨 설정 (학습 시)
        if self.is_train:
            inputs["labels"] = inputs["input_ids"].clone()
            
        return inputs

def custom_collate_fn(batch):
    # 배치 내 텐서들을 패딩하여 묶어주는 함수
    input_ids = torch.nn.utils.rnn.pad_sequence([item['input_ids'] for item in batch], batch_first=True, padding_value=0)
    attention_mask = torch.nn.utils.rnn.pad_sequence([item['attention_mask'] for item in batch], batch_first=True, padding_value=0)
    
    batch_dict = {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'pixel_values': torch.cat([item['pixel_values'] for item in batch], dim=0),
        'image_grid_thw': torch.cat([item['image_grid_thw'] for item in batch], dim=0),
    }
    
    if 'labels' in batch[0]:
        labels = torch.nn.utils.rnn.pad_sequence([item['labels'] for item in batch], batch_first=True, padding_value=-100)
        batch_dict['labels'] = labels
        
    return batch_dict

In [ ]:
# 1. 모델 및 프로세서 로드 (양자화 제거, A100 최적화)
model_id = "Qwen/Qwen2-VL-7B-Instruct"

processor = AutoProcessor.from_pretrained(model_id)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map="auto"
)

# 2. LoRA 설정 (모든 선형 레이어 타겟팅 및 파라미터 스케일업)
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 3. 데이터 로드 (본인 구글 드라이브 마운트 경로에 맞게 수정하세요)
train_df = pd.read_csv('./train.csv')
dev_df = pd.read_csv('./dev.csv') # 검증용으로 일부 사용하거나 테스트용으로 사용

# 빠른 실험을 위해 Train 80% / Val 20% 분할
train_sample = train_df.sample(frac=0.8, random_state=42)
val_sample = train_df.drop(train_sample.index)

train_dataset = QwenVQADataset(train_sample, './', processor, is_train=True)
val_dataset = QwenVQADataset(val_sample, './', processor, is_train=True)

# 4. 학습 인자 및 Trainer 설정
training_args = TrainingArguments(
    output_dir="./qwen_vqa_results",
    num_train_epochs=3, # 시간 상황에 따라 조절
    per_device_train_batch_size=8, # A100 VRAM 허용치까지 올려보세요 (8~16)
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    bf16=True, # A100 특화
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    optim="adamw_torch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=custom_collate_fn
)

# 학습 시작
trainer.train()

# 모델 저장
trainer.save_model("./best_qwen_vqa_model")

In [ ]:
from tqdm import tqdm

# 평가 모드 전환
model.eval()

# 테스트 데이터셋 로드 (is_train=False)
test_dataset = QwenVQADataset(dev_df, './', processor, is_train=False)

predictions = []

with torch.no_grad():
    for idx in tqdm(range(len(test_dataset))):
        inputs = test_dataset[idx]
        
        # 텐서를 GPU로 이동하고 배치 차원 추가
        input_ids = inputs['input_ids'].unsqueeze(0).to(model.device)
        attention_mask = inputs['attention_mask'].unsqueeze(0).to(model.device)
        pixel_values = inputs['pixel_values'].to(model.device) # 이미지 텐서는 배치 차원 추가 X
        image_grid_thw = inputs['image_grid_thw'].to(model.device)
        
        # 생성
        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            image_grid_thw=image_grid_thw,
            max_new_tokens=5, # 정답 알파벳만 나오도록 짧게 설정
            pad_token_id=processor.tokenizer.pad_token_id
        )
        
        # 입력 프롬프트를 제외한 생성된 텍스트만 추출
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(input_ids, generated_ids)
        ]
        
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()
        
        # 정규식을 활용하여 a, b, c, d 중 하나만 추출 (후처리 안전장치)
        match = re.search(r'[a-d]', output_text.lower())
        final_answer = match.group(0) if match else "a" # 못 찾으면 임의로 a 할당
        
        predictions.append(final_answer)

# 제출 파일 생성
submission = dev_df[['id']].copy()
submission['answer'] = predictions
submission.to_csv('submission_qwen_vqa.csv', index=False)
print("추론 완료! submission_qwen_vqa.csv 파일이 생성되었습니다.")